# PEFT tutorial: BERT-base + LoRA, przewidywanie diagnozy z opisu objawów

Powtórzenie tutoriala (punkt C) we własnym środowisku. Zbiór `gretelai/symptom_to_diagnosis`: opis objawów pacjenta → jedna z 22 diagnoz.

In [1]:
import numpy as np
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

ds = load_dataset("gretelai/symptom_to_diagnosis")
labels = sorted(set(ds["train"]["output_text"]))
label2id = {l: i for i, l in enumerate(labels)}
print(ds, len(labels))
ds["train"][0]

/home/shinji/studia/nlp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Generating train split:   0%|          | 0/853 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 853/853 [00:00<00:00, 67147.28 examples/s]

Generating test split:   0%|          | 0/212 [00:00<?, ? examples/s]

Generating test split: 100%|██████████| 212/212 [00:00<00:00, 177554.40 examples/s]

DatasetDict({
    train: Dataset({
        features: ['output_text', 'input_text'],
        num_rows: 853
    })
    test: Dataset({
        features: ['output_text', 'input_text'],
        num_rows: 212
    })
}) 22


{'output_text': 'cervical spondylosis',
 'input_text': "I've been having a lot of pain in my neck and back. I've also been having trouble with my balance and coordination. I've been coughing a lot and my limbs feel weak."}

In [2]:
MODEL = "bert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)


def preprocess(batch):
    enc = tok(batch["input_text"], truncation=True, max_length=128)
    enc["labels"] = [label2id[l] for l in batch["output_text"]]
    return enc


ds = ds.map(preprocess, batched=True, remove_columns=["input_text", "output_text"])

Map:   0%|          | 0/853 [00:00<?, ? examples/s]

Map: 100%|██████████| 853/853 [00:00<00:00, 47703.22 examples/s]

Map:   0%|          | 0/212 [00:00<?, ? examples/s]

Map: 100%|██████████| 212/212 [00:00<00:00, 38215.25 examples/s]

In [3]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(labels), id2label=dict(enumerate(labels)), label2id=label2id)
model = get_peft_model(model, LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1, target_modules=["query", "value"]))
model.print_trainable_parameters()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11328.26it/s]


[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 311,830 || all params: 109,810,988 || trainable%: 0.2840


In [4]:
def metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, preds), "f1": f1_score(p.label_ids, preds, average="macro")}


trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="outputs/tutorial",
        learning_rate=1e-3,
        num_train_epochs=10,
        per_device_train_batch_size=16,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        report_to="none",
    ),
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    data_collator=DataCollatorWithPadding(tok),
    compute_metrics=metrics,
)
trainer.train()
trainer.evaluate()

/home/shinji/studia/nlp/.venv/lib/python3.14/site-packages/transformers/integrations/sdpa_attention.py:174: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:498.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,3.036375,2.602620,0.226415,0.143762
2,2.138925,1.537930,0.523585,0.464320
3,1.375987,1.317150,0.570755,0.541882
4,0.908258,0.700192,0.816038,0.801865
5,0.589047,0.606555,0.792453,0.780363
6,0.369907,0.444386,0.867925,0.861639
7,0.251640,0.407206,0.882075,0.876905
8,0.171961,0.404461,0.882075,0.879662
9,0.140506,0.344646,0.900943,0.897326
10,0.101338,0.309207,0.905660,0.903999


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.101338,0.309207,10,0.905660,0.903999


{'eval_loss': 0.30920711159706116,
 'eval_accuracy': 0.9056603773584906,
 'eval_f1': 0.9039988488501074}

In [5]:
model.eval()
for text in ["I have a high fever, chills and my whole body aches.", "My skin is itchy with red patches and some scaly spots."]:
    enc = tok(text, return_tensors="pt").to(model.device)
    print(text, "->", labels[model(**enc).logits.argmax(-1).item()])

I have a high fever, chills and my whole body aches. -> dengue
My skin is itchy with red patches and some scaly spots. -> fungal infection


/home/shinji/studia/nlp/.venv/lib/python3.14/site-packages/transformers/integrations/sdpa_attention.py:174: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:438.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
